# Electricity Uncertainty Calibration

## Role
Evaluates availability, calibration, and sharpness of preserved uncertainty evidence.

## Inputs
`uncertainty_summary.csv`, two Protocol A Phase 5 checkpoint NPZs, and validated Protocol A/B point forecasts.

## Outputs
Evidence inventory, protocol calibration, coverage error, sharpness, cross-protocol analysis, missing-evidence documentation, provenance, and audit. No forecasts are generated.

## Depends On
`13_Electricity_Foundation_Models.ipynb` · `14_Electricity_Model_Validation_Audit.ipynb` · `15_Electricity_Robustness.ipynb`

## Authoritative Status
**AUTHORITATIVE ELECTRICITY UNCERTAINTY-CALIBRATION EVIDENCE**

## What This Notebook Does Not Do
No training, forecast/interval regeneration, Trust Scores, robustness, significance testing, uncertainty retrofitting, or universal superiority claim.


## 1. Objective and Research Questions

> What uncertainty evidence exists, and how calibrated and informative is it under Protocol A and Protocol B?

### Q1 — Availability
Which models have defensible evidence?

### Q2 — Calibration
How close is coverage to nominal?

### Q3 — Sharpness
How wide are intervals?

### Q4 — Accuracy vs Uncertainty
Does point accuracy imply calibration?

### Q5 — Protocol Sensitivity
Does day-ahead operation change uncertainty quality?

### Q6 — Missing Evidence
How should absence be represented without calling it poor performance?


## 2. Setup


In [ ]:
from pathlib import Path
import hashlib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
def root(p):
 for q in [p.resolve(),*p.resolve().parents]:
  if (q/'results/electricity').is_dir(): return q
ROOT=root(Path.cwd()); R=ROOT/'results/electricity'; S=R/'uncertainty_summary.csv'; PA=R/'protocol_a_validated_forecasts.csv'; PB=R/'protocol_b_validated_forecasts.csv'
V={'Chronos_Bolt_Tiny':R/'.phase5_chronos_a_checkpoint.npz','TimesFM':R/'.phase5_timesfm_a_checkpoint.npz'}
F={'Naive':'Baseline','Daily_Seasonal_Naive':'Baseline','Weekly_Seasonal_Naive':'Baseline','Moving_Average':'Baseline','ARIMA':'Statistical','SARIMA':'Statistical','Prophet':'Statistical','Simple_Exponential_Smoothing':'Statistical','Holt_Winters':'Statistical','DHR_ARIMA':'Statistical','LSTM':'Neural','Chronos_Bolt_Tiny':'Foundation','TimesFM':'Foundation'}
L={'Chronos_Bolt_Tiny':'Chronos-Bolt-Tiny','TimesFM':'TimesFM'}; SCALE=117.057971280678
u=pd.read_csv(S); pa=pd.read_csv(PA); pb=pd.read_csv(PB); NOM=float(u.loc[u.Available,'Nominal_Coverage'].unique().item())
def enrich(g):
 g=g.copy(); g['ACE']=g.Empirical_Coverage-g.Nominal_Coverage; g['Abs_ACE']=g.ACE.abs(); g['Direction']=np.where(g.ACE>0,'overcoverage','undercoverage'); return g
def show(g,d=6): display(g.round(d))
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest()
a=enrich(u[u.Available]); missing=u[~u.Available]; models=list(dict.fromkeys(u.Model)); plt.style.use('seaborn-v0_8-whitegrid')


## 3. Uncertainty Evidence Framework

### 3.1 Evidence Taxonomy


In [ ]:
display(pd.DataFrame([['Native probabilistic','Direct model intervals',', '.join(L[m] for m in a.Model.unique()),'Available'],['Calibrated native','Separately calibrated output','None','Not preserved'],['Retrofitted empirical','Independent pre-test residual interval','None','Not created'],['Unavailable','No defensible interval',', '.join(L.get(m,m) for m in missing.Model.unique()),'Missing, not poor calibration']],columns=['Evidence Type','Meaning','Models','Interpretation']))


### 3.2 Nominal Coverage
The single nominal level is 80%; ideal marginal coverage is approximately 0.80.

### 3.3 Calibration Metrics
PICP is the fraction inside bounds. ACE is empirical minus nominal coverage; negative means undercoverage and positive overcoverage. Absolute ACE is distance from nominal. Average width measures sharpness. A proper interval score combines width with miss penalties.

Protocol A exact bounds exist in saved NPZs, permitting a proper 80% interval score. Protocol B retains aggregate coverage and width only, so no proper score is reconstructed.

### 3.4 Coverage vs Sharpness
Wide intervals may be uninformative; narrow intervals may be overconfident. No composite is introduced.

### 3.5 Comparability Rules
Use the same nominal level and scale, keep protocols separate, treat missing as missing, distinguish accuracy from calibration, and do not infer full distributional calibration from one level.


## 4. Evidence Availability Across Models

### 4.1 Full Model Inventory — Table


In [ ]:
rr=[]
for m in models:
 g=u[u.Model==m].set_index('Protocol'); rr.append({'Model':L.get(m,m),'Family':F[m],'Protocol A':'Available' if g.loc['A','Available'] else 'Unavailable','Protocol B':'Available' if g.loc['B','Available'] else 'Unavailable','Evidence Type':g.Evidence_Type.iloc[0],'Availability':'Available' if g.Available.any() else 'Unavailable','Note':g.Notes.iloc[0]})
inventory=pd.DataFrame(rr); display(inventory)


### 4.2 Availability Summary


In [ ]:
availability=inventory.Availability.value_counts().rename_axis('Status').reset_index(name='Model Count'); availability['Share']=availability['Model Count']/len(inventory); show(availability)


### 4.3 Evidence Availability Visualization

Unavailable is neutral, not a low score.


In [ ]:
mat=u.pivot(index='Model',columns='Protocol',values='Available').reindex(models).astype(int); fig,ax=plt.subplots(figsize=(6,6)); ax.imshow(mat,cmap=plt.cm.colors.ListedColormap(['#ddd','#009E73'])); ax.set_xticks([0,1],['A','B']); ax.set_yticks(range(len(mat)),[L.get(m,m) for m in mat.index])
for i in range(len(mat)):
 for j in range(2): ax.text(j,i,'Available' if mat.iloc[i,j] else 'Unavailable',ha='center',va='center',fontsize=7)
ax.set_title('Uncertainty evidence availability'); plt.show()


## 5. Protocol A — Uncertainty Calibration

### 5.1 Available Evidence — Table


In [ ]:
cal_a=a[a.Protocol=='A'].copy().sort_values('Abs_ACE'); cal_a['Model']=cal_a.Model.map(L); show(cal_a[['Model','Nominal_Coverage','Empirical_Coverage','ACE','Abs_ACE','Average_Width']])


### 5.2 Coverage Calibration — Visualization


In [ ]:
g=cal_a; ax=g.plot.bar(x='Model',y='Empirical_Coverage',legend=False); ax.axhline(NOM,color='black',linestyle='--'); ax.set(title='Protocol A: coverage',ylabel='Coverage',ylim=(0,1),xlabel=''); plt.xticks(rotation=0); from pathlib import Path
_figure_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "figures").is_dir())
_figure_dir = _figure_root / "figures" / "electricity"
_figure_dir.mkdir(parents=True, exist_ok=True)
_figure_path = _figure_dir / "electricity_protocol_a_uncertainty_coverage.png"
plt.gcf().savefig(_figure_path, dpi=300, bbox_inches="tight")
plt.show()


### 5.3 Absolute Coverage Error — Visualization


In [ ]:
ax=cal_a.plot.bar(x='Model',y='Abs_ACE',legend=False,color='#D55E00'); ax.set(title='Protocol A: absolute ACE',ylabel='Lower is closer',xlabel=''); plt.xticks(rotation=0); plt.show()


### 5.4 Interval Width — Visualization

Sharpness must be interpreted with coverage.


In [ ]:
ax=cal_a.plot.bar(x='Model',y='Average_Width',legend=False,color='#009E73'); ax.set(title='Protocol A: average width',ylabel='MW',xlabel=''); plt.xticks(rotation=0); plt.show()


### 5.5 Coverage–Sharpness Trade-Off


In [ ]:
g=cal_a; fig,ax=plt.subplots(); ax.scatter(g.Average_Width,g.Abs_ACE,s=80)
for _,r in g.iterrows(): ax.annotate(r.Model,(r.Average_Width,r.Abs_ACE))
ax.set(title='Protocol A: coverage–sharpness',xlabel='Width (MW)',ylabel='Absolute ACE'); plt.show()


### 5.6 Protocol A Model Interpretation


In [ ]:
t=cal_a[['Model','Abs_ACE','Average_Width','Direction']].copy(); t['Calibration Rank']=t.Abs_ACE.rank().astype(int); t['Sharpness Rank']=t.Average_Width.rank().astype(int); display(t.sort_values('Calibration Rank'))


## 6. Protocol B — Uncertainty Calibration

### 6.1 Available Evidence — Table


In [ ]:
cal_b=a[a.Protocol=='B'].copy().sort_values('Abs_ACE'); cal_b['Model']=cal_b.Model.map(L); show(cal_b[['Model','Nominal_Coverage','Empirical_Coverage','ACE','Abs_ACE','Average_Width']])


### 6.2 Coverage Calibration — Visualization


In [ ]:
g=cal_b; ax=g.plot.bar(x='Model',y='Empirical_Coverage',legend=False); ax.axhline(NOM,color='black',linestyle='--'); ax.set(title='Protocol B: coverage',ylabel='Coverage',ylim=(0,1),xlabel=''); plt.xticks(rotation=0); from pathlib import Path
_figure_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "figures").is_dir())
_figure_dir = _figure_root / "figures" / "electricity"
_figure_dir.mkdir(parents=True, exist_ok=True)
_figure_path = _figure_dir / "electricity_protocol_b_uncertainty_coverage.png"
plt.gcf().savefig(_figure_path, dpi=300, bbox_inches="tight")
plt.show()


### 6.3 Absolute Coverage Error — Visualization


In [ ]:
ax=cal_b.plot.bar(x='Model',y='Abs_ACE',legend=False,color='#D55E00'); ax.set(title='Protocol B: absolute ACE',ylabel='Lower is closer',xlabel=''); plt.xticks(rotation=0); plt.show()


### 6.4 Interval Width — Visualization

Sharpness must be interpreted with coverage.


In [ ]:
ax=cal_b.plot.bar(x='Model',y='Average_Width',legend=False,color='#009E73'); ax.set(title='Protocol B: average width',ylabel='MW',xlabel=''); plt.xticks(rotation=0); plt.show()


### 6.5 Coverage–Sharpness Trade-Off


In [ ]:
g=cal_b; fig,ax=plt.subplots(); ax.scatter(g.Average_Width,g.Abs_ACE,s=80)
for _,r in g.iterrows(): ax.annotate(r.Model,(r.Average_Width,r.Abs_ACE))
ax.set(title='Protocol B: coverage–sharpness',xlabel='Width (MW)',ylabel='Absolute ACE'); plt.show()


### 6.6 Protocol B Model Interpretation


In [ ]:
t=cal_b[['Model','Abs_ACE','Average_Width','Direction']].copy(); t['Calibration Rank']=t.Abs_ACE.rank().astype(int); t['Sharpness Rank']=t.Average_Width.rank().astype(int); display(t.sort_values('Calibration Rank'))


## 7. Cross-Protocol Calibration Analysis

### 7.1 Coverage Stability


In [ ]:
cross=a.pivot(index='Model',columns='Protocol',values=['Empirical_Coverage','Abs_ACE','Average_Width']); coverage=pd.DataFrame({'Model':[L[m] for m in cross.index],'Coverage A':cross.Empirical_Coverage.A.values,'Coverage B':cross.Empirical_Coverage.B.values,'Difference B-A':(cross.Empirical_Coverage.B-cross.Empirical_Coverage.A).values,'Abs ACE A':cross.Abs_ACE.A.values,'Abs ACE B':cross.Abs_ACE.B.values}); show(coverage)


### 7.2 Width Stability


In [ ]:
width=pd.DataFrame({'Model':[L[m] for m in cross.index],'Width A':cross.Average_Width.A.values,'Width B':cross.Average_Width.B.values,'Width Ratio B/A':(cross.Average_Width.B/cross.Average_Width.A).values}); show(width)


### 7.3 Protocol Sensitivity — Visualization


In [ ]:
fig,ax=plt.subplots()
for _,r in coverage.iterrows(): ax.plot([0,1],[r['Abs ACE A'],r['Abs ACE B']],marker='o',label=r.Model)
ax.set_xticks([0,1],['A','B']); ax.set_ylabel('Absolute ACE'); ax.legend(); from pathlib import Path
_figure_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "figures").is_dir())
_figure_dir = _figure_root / "figures" / "electricity"
_figure_dir.mkdir(parents=True, exist_ok=True)
_figure_path = _figure_dir / "electricity_uncertainty_protocol_sensitivity.png"
plt.gcf().savefig(_figure_path, dpi=300, bbox_inches="tight")
plt.show()


### 7.4 Calibration Direction Stability


In [ ]:
d=a.pivot(index='Model',columns='Protocol',values='Direction').reset_index(); d.Model=d.Model.map(L); display(d)


## 8. Point Accuracy vs Uncertainty Quality


In [ ]:
ar=[]
for p,f in [('A',pa),('B',pb)]:
 for m in a.Model.unique(): ar.append({'Protocol':p,'Model':m,'Point MASE-48':np.mean(np.abs(f.Actual-f[m]))/SCALE})
accuracy=pd.DataFrame(ar); accuracy['Point Rank']=accuracy.groupby('Protocol')['Point MASE-48'].rank().astype(int); trade=accuracy.merge(a[['Protocol','Model','Empirical_Coverage','Abs_ACE','Average_Width']],on=['Protocol','Model']); trade.Model=trade.Model.map(L); show(trade)
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,(p,g) in zip(axes,trade.groupby('Protocol')):
 ax.scatter(g['Point MASE-48'],g.Abs_ACE,s=80)
 for _,r in g.iterrows(): ax.annotate(r.Model,(r['Point MASE-48'],r.Abs_ACE))
 ax.set(title=f'Protocol {p}',xlabel='Point MASE-48',ylabel='Absolute ACE')
from pathlib import Path
_figure_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "figures").is_dir())
_figure_dir = _figure_root / "figures" / "electricity"
_figure_dir.mkdir(parents=True, exist_ok=True)
_figure_path = _figure_dir / "electricity_accuracy_uncertainty_tradeoff.png"
plt.gcf().savefig(_figure_path, dpi=300, bbox_inches="tight")
plt.show()


This is a two-dimensional trade-off, not a composite score.


## 9. Missing Uncertainty Evidence

### 9.1 Models Without Preserved Uncertainty


In [ ]:
display(missing.assign(Family=missing.Model.map(F),Status='Unavailable')[['Model','Family','Protocol','Status','Notes']].rename(columns={'Notes':'Reason'}))


### 9.2 Why Missing Evidence Is Not Poor Calibration
No interval means no calibration score; it implies neither zero, perfect, nor poor quality.

### 9.3 Downstream Handling
Notebook 17 may separate missing-evidence penalisation from evidence-available scoring. Neither is calculated here.


## 10. Reproducibility and Artifact Provenance

### 10.1 Uncertainty Artifact Inventory


In [ ]:
spec=[(S,'Aggregate calibration','A,B'),(V['Chronos_Bolt_Tiny'],'Chronos A bounds','A'),(V['TimesFM'],'TimesFM A bounds','A'),(PA,'A points','A'),(PB,'B points','B')]; out=[]
for p,purpose,protocol in spec:
 if p.suffix=='.csv': q=pd.read_csv(p); count=len(q); cols=', '.join(q.columns)
 else:
  with np.load(p) as q: count=len(q[q.files[0]]); cols=', '.join(q.files)
 out.append({'Artifact':p.name,'Purpose':purpose,'Rows':count,'Columns/Arrays':cols,'Protocols':protocol,'Status':'Frozen; read only','SHA-256':sha(p)})
artifacts=pd.DataFrame(out); display(artifacts)


### 10.2 Row-Level Evidence Availability


In [ ]:
display(pd.DataFrame([['Lower/upper vectors','Protocol A only','A proper score possible'],['Point vectors','A and B','Accuracy comparison possible'],['Aggregate coverage/width','A and B','ACE/sharpness reproducible'],['Horizon intervals','No','Horizon calibration unavailable'],['Protocol B interval score','No','Not calculated']],columns=['Component','Available?','Consequence']))
def iscore(y,l,h,alpha): return (h-l)+(2/alpha)*(l-y)*(y<l)+(2/alpha)*(y-h)*(y>h)
sr=[]
for m,p in V.items():
 with np.load(p) as q: lo=q['lo']; hi=q['hi']
 y=pa.Actual.to_numpy(); ref=a[(a.Protocol=='A')&(a.Model==m)].iloc[0]; cv=np.mean((y>=lo)&(y<=hi)); wd=np.mean(hi-lo); sr.append({'Model':L[m],'Rows':len(y),'Vector coverage':cv,'Stored coverage':ref.Empirical_Coverage,'Coverage difference':cv-ref.Empirical_Coverage,'Vector width':wd,'Stored width':ref.Average_Width,'Width difference':wd-ref.Average_Width,'80% interval score':np.mean(iscore(y,lo,hi,1-NOM))})
scores=pd.DataFrame(sr); show(scores,10)


### 10.3 Regeneration Boundary
Protocol B bounds, horizon intervals, or another nominal level would require new inference. None is regenerated.


## 11. Validation and Evidence Integrity


In [ ]:
rows=[]
def add(c,k,e,o,s): rows.append({'Category':c,'Check':k,'Expected':e,'Observed':o,'Status':s})
add('Loading','Artifact/schema',True,S.exists() and len(u.columns)==9,'PASS'); add('Availability','Expected models',{'Chronos_Bolt_Tiny','TimesFM'},set(a.Model.unique()),'PASS' if set(a.Model.unique())=={'Chronos_Bolt_Tiny','TimesFM'} else 'FAIL'); add('Availability','Missing notes',True,missing.Notes.notna().all(),'PASS'); add('Calibration','Complete summaries',True,not a[['Nominal_Coverage','Empirical_Coverage','Average_Width']].isna().any().any(),'PASS'); add('Calibration','Coverage ranges',True,a.Nominal_Coverage.between(0,1).all() and a.Empirical_Coverage.between(0,1).all(),'PASS'); add('Calibration','Widths valid',True,np.isfinite(a.Average_Width).all() and a.Average_Width.ge(0).all(),'PASS'); add('Calibration','ACE exact',True,np.allclose(a.ACE,a.Empirical_Coverage-a.Nominal_Coverage),'PASS'); add('Calibration','Abs ACE exact',True,np.allclose(a.Abs_ACE,a.ACE.abs()),'PASS'); add('Protocol','A/B integrity',True,set(u.Protocol)=={'A','B'} and a.groupby('Model').Protocol.apply(set).eq({'A','B'}).all(),'PASS'); add('Missing','No fabrication',True,missing[['Nominal_Coverage','Empirical_Coverage','Average_Width']].isna().all().all(),'PASS'); add('Missing','No test retrofit',True,missing.Notes.str.contains('final-test residuals not used').all(),'PASS'); parity=np.allclose(scores['Coverage difference'],0,atol=1e-6) and np.allclose(scores['Width difference'],0,atol=1e-6); add('Source','A vector parity at stored precision',True,parity,'PASS' if parity else 'FAIL'); add('Source','B proper score','Bounds absent','Not calculated','NOT AVAILABLE BY DESIGN'); add('Source','Horizon calibration','Vectors absent','Not calculated','NOT AVAILABLE BY DESIGN')
audit=pd.DataFrame(rows); display(audit); assert audit[audit.Status!='NOT AVAILABLE BY DESIGN'].Status.eq('PASS').all(); order=['PASS','FAIL','NOT AVAILABLE BY DESIGN']; vc=audit.Status.value_counts(); audit_summary=pd.DataFrame({'Status':order,'Count':[int(vc.get(q,0)) for q in order]}); audit_summary.loc[len(audit_summary)]=['TOTAL',len(audit)]; display(audit_summary)


## 12. Key Findings


In [ ]:
ba=cal_a.sort_values('Abs_ACE').iloc[0]; bb=cal_b.sort_values('Abs_ACE').iloc[0]; pw=trade.loc[trade.groupby('Protocol')['Point MASE-48'].idxmin()].set_index('Protocol'); cw=trade.loc[trade.groupby('Protocol').Abs_ACE.idxmin()].set_index('Protocol'); f=[f'{len(a.Model.unique())} of {len(inventory)} models have uncertainty evidence.',f'Protocol A closest: {ba.Model}, absolute ACE {ba.Abs_ACE:.4f}, {ba.Direction}.',f'Protocol B closest: {bb.Model}, absolute ACE {bb.Abs_ACE:.4f}, {bb.Direction}.','Protocol B width ratios: '+', '.join(f"{r.Model} {r['Width Ratio B/A']:.2f}x" for _,r in width.iterrows())+'.',f"Point winners: {pw.loc['A','Model']} and {pw.loc['B','Model']}; calibration winners: {cw.loc['A','Model']} and {cw.loc['B','Model']}.",f'{len(missing.Model.unique())} models have missing evidence, not poor calibration.','Predictive accuracy and probabilistic reliability are distinct.']; display(Markdown(chr(10).join(f'{i+1}. {v}' for i,v in enumerate(f))))


## 13. Limitations
Only two native-uncertainty models; one nominal level; no Protocol B proper score; no horizon calibration; missing point-model evidence; marginal coverage only; width is not a proper score alone; one region/test period; unresolved checkpoint revisions and pretraining overlap.


## 14. Next Notebook
Next: `17_Electricity_Trustworthiness.ipynb`

Notebook 16 establishes uncertainty evidence and boundaries. Notebook 17 synthesizes accuracy, robustness, temporal stability, uncertainty, and transparency/auditability.
